# Part 0 — Healthcare Data Processing and Analytics

**Purpose:** inspect, clean, and describe the supplied synthetic healthcare CSV, and decide what you would need to know before anyone modelled it. **No model is fitted in this notebook.**

Most of the work in a real health data project happens before any model exists. This notebook is that work. Every step ends in a **decision or a question for the data owner**, not just a number.

**Roadmap:** load → audit → clean duplicates → process dates → summarise → compare groups → decide what each column is → write down what you would ask.

> **Teaching boundary:** this is supplied course material. Nothing here is clinical decision support, a validated billing tool, a causal analysis, or deployment guidance.

## 0. Before you begin

**Runtime:** about 30 minutes. **Prerequisites:** none beyond running cells in order.

**What you need:** the file `healthcare_dataset.csv`. The next cell looks for it next to this notebook, one folder up, and at the course path.

**If you are using Colab:** right-click `healthcare_dataset.csv` in the file list and choose **Upload to Colab**, then run this notebook from the top.

**If anything breaks:** restart the kernel and use **Run All**. A restarted kernel forgets every variable, so partial reruns fail in confusing ways. See `troubleshooting.md`.

In [ ]:
from pathlib import Path
import platform

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid", palette="colorblind")
pd.set_option("display.width", 120)

print(f"Python: {platform.python_version()} | pandas: {pd.__version__}")

## 1. Load the supplied course data

A **row** here is one recorded admission. A **column** is one recorded variable.

The code checks several locations so the notebook works whether you run it locally or in Colab.

**Look for:** the source shape, printed as `(rows, columns)`, before any cleaning.

In [ ]:
def find_data_file(filename, max_levels_up=4):
    """Look for a file in the working directory and each folder above it."""
    here = Path.cwd().resolve()
    for directory in [here, *list(here.parents)[:max_levels_up]]:
        candidate = directory / filename
        if candidate.exists():
            return candidate
    return None

DATA_PATH = find_data_file("healthcare_dataset.csv")
if DATA_PATH is None:
    raise FileNotFoundError(
        "Could not find healthcare_dataset.csv. Put it next to this notebook, "
        "or upload it to the Colab runtime, then run this cell again."
    )

raw_df = pd.read_csv(DATA_PATH)

print(f"Loaded: {DATA_PATH.resolve()}")
print(f"Source shape: {raw_df.shape[0]:,} rows x {raw_df.shape[1]} columns")
display(raw_df.head())

### What do we see?

Fifteen columns mixing four different kinds of information: people (`Name`, `Age`, `Gender`), clinical detail (`Medical Condition`, `Medication`, `Test Results`), administration (`Insurance Provider`, `Billing Amount`, `Room Number`), and timing (`Date of Admission`, `Discharge Date`).

Already notice `Bobby JacksOn` and `LesLie TErRy` in the preview. The capitalisation is inconsistent, which tells you this column was not produced by a controlled process. That is a clue about the whole file, not just that column.

This preview confirms the file loaded. It says nothing yet about quality.

## 2. Audit structure and data quality

An **audit** records what is actually in the file: the type of each column, how much is missing, and how many distinct values each column holds.

`unique` is the most informative column here. A variable with very few distinct values behaves like a category. A variable with tens of thousands behaves like an identifier.

**Look for:** the `unique` count for `Name`, `Doctor`, and `Hospital`, and the duplicate-row count at the bottom.

In [ ]:
audit = pd.DataFrame({
    "dtype": raw_df.dtypes.astype(str),
    "missing": raw_df.isna().sum(),
    "missing %": (raw_df.isna().mean() * 100).round(2),
    "unique": raw_df.nunique(),
})
audit["unique % of rows"] = (audit["unique"] / len(raw_df) * 100).round(1)
display(audit)

duplicate_count = raw_df.duplicated().sum()
print(f"Exact duplicate rows: {duplicate_count:,} ({duplicate_count / len(raw_df) * 100:.2f}% of the file)")

### What do we see?

**No missing values anywhere.** Real health data almost never looks like this. Treat it as a property of a synthetic file, not as good news.

**`Name` has close to one distinct value per row.** It behaves like an identifier. Identifiers must never be given to a model: the model would memorise rows rather than learn a pattern, and performance would collapse on anyone new.

**`Doctor` and `Hospital` have very high counts too.** These encode local process — which clinician happened to be on shift, which site recorded the admission. A model can use them, but what it learns will not transfer to another site.

**There are exact duplicate rows.** A duplicate is either the same admission recorded twice or two genuinely identical records. You cannot tell which from the file alone.

> **Question for the data owner #1:** are duplicate rows a recording artifact, or can two admissions genuinely share all fifteen values?

## 3. Remove exact duplicates and process dates

Remove exact duplicates before summarising, so repeated records do not distort counts and averages.

Convert the two date columns with `errors="coerce"`, so anything unparseable becomes missing rather than being silently guessed. Then derive length of stay.

**Look for:** how many rows were removed, and whether any date failed to parse.

In [ ]:
df = raw_df.drop_duplicates().copy()

for column in ["Date of Admission", "Discharge Date"]:
    df[column] = pd.to_datetime(df[column], errors="coerce")

failed_to_parse = df[["Date of Admission", "Discharge Date"]].isna().sum()
df["Length of Stay (days)"] = (df["Discharge Date"] - df["Date of Admission"]).dt.days

print(f"Removed {len(raw_df) - len(df):,} exact duplicates; {len(df):,} rows retained.")
print(f"Dates that failed to parse:\n{failed_to_parse.to_string()}")
display(df[["Date of Admission", "Discharge Date", "Length of Stay (days)"]].head())

### What do we see?

Every date parsed and every stay has a length. Again, cleaner than real data.

**Length of stay is a derived column, and it is important to be clear about when it exists.** It can only be calculated after discharge. If you were predicting something at the moment a patient arrives, this number does not exist yet.

That distinction — *when does this value become known?* — is the single most useful habit in this notebook. Section 6 applies it to every column.

## 4. Describe the numeric variables

**Descriptive statistics** report centre, spread, and range. They describe this file. They do not explain why any value is what it is.

Billing is deliberately left exactly as recorded, including any non-positive values.

**Look for:** the `min` for `Billing Amount`, and the `non-positive count` column.

In [ ]:
numeric_columns = ["Age", "Room Number", "Billing Amount", "Length of Stay (days)"]

numeric_summary = df[numeric_columns].describe().T
numeric_summary["non-positive count"] = [(df[column] <= 0).sum() for column in numeric_columns]
display(numeric_summary.round(2))

### Plot the same two variables

A table gives you the summary statistics; a plot shows you the shape. Both are needed, because very different distributions can share a mean and a standard deviation.

**Look for:** values to the left of the dashed zero line in the left panel, and the overall shape of the right panel.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

sns.histplot(df["Billing Amount"], bins=40, kde=True, ax=axes[0])
axes[0].axvline(0, color="black", linestyle="--")
axes[0].set_title("Billing Amount distribution")
axes[0].set_xlabel("Billing amount")

sns.histplot(df["Length of Stay (days)"].dropna(), bins=30, ax=axes[1])
axes[1].set_title("Length of Stay distribution")
axes[1].set_xlabel("Days")

plt.tight_layout()
plt.show()

### What do we see?

**Some billing amounts are negative.** The dashed line marks zero so they are visible.

A negative bill is not automatically an error. It could be a refund, a reversed charge, a credit adjustment, or a data-entry mistake. Those four possibilities call for four different responses, and **you cannot choose between them from this file.**

Deleting them would be the most common mistake here. It silently changes what the dataset represents and hides a question you have not answered.

**Length of stay is roughly uniform.** Real length-of-stay distributions are strongly right-skewed — most stays short, a long tail of extended ones. A flat distribution is a signature of synthetic generation, and it means any model trained here would not face the pattern real data presents.

> **Question for the data owner #2:** what does a negative billing amount represent in the source system, and should those records be included?

> **Question for the data owner #3:** was this file generated synthetically, and if so, which relationships between columns were preserved?

## 5. Summarise the recorded categories and groups

Counts and group summaries are descriptive. A balanced outcome does not mean the outcome can be predicted, and a difference between groups does not establish a cause.

**Look for:** how evenly `Test Results` is split, and whether billing differs by admission type.

In [ ]:
for column in ["Admission Type", "Insurance Provider", "Medical Condition", "Test Results"]:
    counts = df[column].value_counts(dropna=False).to_frame("count")
    counts["percent"] = (counts["count"] / len(df) * 100).round(1)
    print(f"\n{column}")
    display(counts)

### Compare billing across admission types

Group the billing amount by admission type and inspect both the table and the spread.

**Look for:** whether emergency admissions cost more than elective ones. Write down what you expect *before* reading the numbers.

In [ ]:
billing_by_type = df.groupby("Admission Type")["Billing Amount"].agg(
    n="size", mean="mean", median="median", std="std"
)
display(billing_by_type.round(2))

fig, ax = plt.subplots(figsize=(8, 4))
sns.boxplot(data=df, x="Admission Type", y="Billing Amount", ax=ax)
ax.set_title("Billing Amount by Admission Type")
plt.tight_layout()
plt.show()

### What do we see?

**`Test Results` is split almost exactly into thirds.** In real clinical data, outcome categories are rarely balanced.

This matters more than it looks. An even split means a model that always guesses one category would be right about a third of the time — so a third is the bar any real model must clear.

**Billing barely differs across admission types.** Emergency, urgent, and elective admissions have nearly identical means and medians.

Clinically, that is implausible: emergency care generally costs more. When a variable that *should* be related to cost turns out not to be, the likely explanation is that the values were generated independently of each other rather than reflecting a real process.

This is the observation that decides what the rest of Lab 2 does with this file.

## 6. Decide what each column is, and when it becomes known

Before selecting any feature, fix a **prediction time**: the moment the prediction would actually be made. Here we use **admission**.

**Leakage** is when a workflow uses information that would not be available at the prediction time, or lets evaluation data influence training. It produces impressive results that collapse in practice. It comes in two forms:

1. **Future or outcome leakage** — using `Discharge Date` to predict something at admission.
2. **Preprocessing leakage** — fitting a scaler, imputer, or encoder on all rows before splitting.

Parts 1A and 1B deal with the second form using pipelines. This section deals with the first.

**Look for:** which columns fall outside "available at admission".

> ### Your turn
>
> The next code cell is incomplete. Replace every `# TODO` with working code.
> The surrounding explanation tells you what the cell must do and what to look for.
> If you get stuck, the reasoning you need is in the Markdown directly above.

In [ ]:
# TODO: classify each group of columns by when its value becomes known.
# Use exactly one of these labels in the second position:
#   "Potentially available" | "Identifier" | "Operational"
#   "Timing ambiguous"      | "Future information" | "Outcome" | "Recorded outcome"
# Prediction time for this exercise is ADMISSION.

feature_policy = pd.DataFrame(
    [
        ("Age, Gender, Blood Type, Medical Condition, Insurance Provider, Admission Type",
         "TODO", "TODO: why?"),
        ("Name", "TODO", "TODO: why?"),
        ("Doctor, Hospital, Room Number", "TODO", "TODO: why?"),
        ("Medication", "TODO", "TODO: why?"),
        ("Discharge Date, Length of Stay", "TODO", "TODO: why?"),
        ("Billing Amount", "TODO", "TODO: why?"),
        ("Test Results", "TODO", "TODO: why?"),
    ],
    columns=["Fields", "Status at admission", "Reason"],
)
display(feature_policy)

### What do we see?

Only six of fifteen columns survive as admission-time inputs, and even those carry conditions.

Two deserve particular care. `Insurance Provider` is available administratively, but it can encode access to care and structural inequity rather than anything clinical. `Gender` is available, but any model using it needs subgroup error checking before anyone reads meaning into it.

Notice how much of the file disappeared. This is normal. **A column existing in a CSV is not the same as a column being usable for a stated question.**

## 7. Putting it together: what would you tell a colleague?

You now have enough to answer the question a colleague would actually ask: *can we build something useful on this?*

The honest summary for this file is: **you can practise the workflow on it, but you should not benchmark a model on it.** The columns available at admission carry very little information about the recorded outcomes, and several distributions do not behave like the real processes they represent.

That conclusion is a finding, not a failure. Recognising it early is the skill being taught. A team that skipped this notebook would have trained a model, seen it perform near chance, and spent a week looking for a bug in the code.

**This is why Parts 1A and 1B use scikit-learn's bundled datasets** — so the mechanics of a correct workflow can be taught on data where a signal genuinely exists, without pretending this file contains one.

In [ ]:
summary = pd.DataFrame([
    {"finding": "No missing values",
     "implication": "Unusual for health data; do not assume future data will look like this."},
    {"finding": f"{duplicate_count:,} exact duplicate rows",
     "implication": "Removed before summaries; ask whether they are a recording artifact."},
    {"finding": "Negative billing amounts present",
     "implication": "Kept as recorded; needs a source-system explanation before any decision."},
    {"finding": "Test Results split evenly into thirds",
     "implication": "A trivial guess is right about one third of the time; that is the bar to beat."},
    {"finding": "Billing nearly identical across admission types",
     "implication": "Suggests columns were generated independently; weak basis for a cost model."},
    {"finding": "Only 6 of 15 columns available at admission",
     "implication": "Most of the file cannot be used for an admission-time question."},
])
display(summary)

## 8. Try it yourself

Pick one and write two or three sentences.

1. Group `Billing Amount` by `Medical Condition` instead of `Admission Type`. Does any condition stand out? Would you expect it to?
2. Check whether `Length of Stay` differs by `Admission Type`. State what you would expect clinically, then compare it with what the data shows.
3. Take one field you placed in "Potentially available" and write the exact question you would ask the data owner to confirm its timing.

Do not draw a clinical conclusion from any of these. Describe the pattern, then state what you would need in order to explain it.

## 9. What's next, and where to get help

- `Part-1A_Classification_ML_Pipeline.ipynb` — predicting a category, with preprocessing for categorical health columns.
- `Part-1B_Regression_ML_Pipeline.ipynb` — predicting a number, including a worked demonstration of preprocessing leakage.
- `../handouts/` — one-page references for the pipeline, data splitting, metric choice, and a leakage checklist.
- `glossary_and_quick_reference.md` — terms and metric directions.
- `troubleshooting.md` — read this before changing code.

## Summary

This CSV supports reproducible cleaning and descriptive analysis. Under the stated admission-time framing it does not support clinical decisions, causal conclusions, or a useful predictive benchmark.

- [x] Audited types, missingness, uniqueness, and duplicates.
- [x] Removed exact duplicates and processed dates.
- [x] Described numeric variables without deleting inconvenient values.
- [x] Compared groups without claiming causation.
- [x] Classified every column by when it becomes known.
- [x] Wrote down three questions for the data owner.

**Limits:** everything above describes *this file*. It does not establish what is true of the population the file claims to represent, it does not explain why any value is what it is, and it does not support a causal or clinical conclusion. Descriptive statistics describe; they do not explain.